# Foraging Cache — DuckDB Query Examples

The parquet database lives at:
- `scratch/foraging_cache/session_table_sample.parquet` — flat session table
- `scratch/foraging_cache/trial_table/` — Hive-partitioned by `subject_id`
- `scratch/foraging_cache/event_table/` — Hive-partitioned by `subject_id`

**Most common pattern**: filter the session table, then fetch trials/events for those
sessions with session-level metadata already merged in.

Three DuckDB options are needed when reading across sessions:
- `hive_partitioning=true` — enables partition-level filtering on `subject_id`
- `union_by_name=true` — merges column schemas across files; different NWB readers produce different column sets, missing columns fill with NULL
- `CAST(subject_id AS VARCHAR)` — DuckDB infers the `subject_id=<n>` partition directory name as BIGINT; cast to VARCHAR to compare against the session table

In [ ]:
import duckdb
import pandas as pd

SCRATCH      = "/root/capsule/scratch/foraging_cache"
SESSION_OUT  = f"{SCRATCH}/session_table_sample.parquet"
TRIAL_OUT    = f"{SCRATCH}/trial_table"
EVENT_OUT    = f"{SCRATCH}/event_table"

# Reusable snippets — union_by_name merges schemas across files from different NWB readers
READ_TRIALS = f"read_parquet('{TRIAL_OUT}/**/*.parquet', hive_partitioning=true, union_by_name=true)"
READ_EVENTS = f"read_parquet('{EVENT_OUT}/**/*.parquet', hive_partitioning=true, union_by_name=true)"

## 1. Browse the session table

In [ ]:
duckdb.sql(f"""
    SELECT _session_id, subject_id, session_date, task, finished_trials, foraging_eff
    FROM read_parquet('{SESSION_OUT}')
    ORDER BY session_date DESC
    LIMIT 10
""").df()

In [ ]:
duckdb.sql(f"""
    SELECT _session_id, subject_id, session_date, task, finished_trials, foraging_eff
    FROM read_parquet('{SESSION_OUT}')
    ORDER BY session_date DESC
    LIMIT 10
""").df()

## 2. Primary use case — filter sessions → fetch trials with session keys merged

Filter the session table however you like, then JOIN to the trial table so every trial
row already carries the session-level metadata (subject, date, task, foraging_eff, …).

In [ ]:
df_trials = duckdb.sql(f"""
    WITH sel AS (
        SELECT _session_id, subject_id, session_date, task, foraging_eff
        FROM read_parquet('{SESSION_OUT}')
        WHERE task LIKE '%Uncoupled%'
          AND foraging_eff > 0.8
    )
    SELECT
        s.subject_id,
        s.session_date,
        s.task,
        s.foraging_eff,
        t.session_id,
        t.animal_response,
        t.earned_reward,
        t.reward_probabilityL,
        t.reward_probabilityR,
        t.rewarded_historyL,
        t.rewarded_historyR
    FROM {READ_TRIALS} t
    JOIN sel s ON t.session_id = s._session_id
    WHERE CAST(t.subject_id AS VARCHAR) IN (SELECT subject_id FROM sel)
    ORDER BY s.subject_id, s.session_date
""").df()

print(f"{len(df_trials)} trials from {df_trials['session_id'].nunique()} sessions")
df_trials.head(10)

## 3. Same pattern for events

In [ ]:
df_events = duckdb.sql(f"""
    WITH sel AS (
        SELECT _session_id, subject_id, session_date, task, foraging_eff
        FROM read_parquet('{SESSION_OUT}')
        WHERE task LIKE '%Uncoupled%'
          AND foraging_eff > 0.8
    )
    SELECT
        s.subject_id,
        s.session_date,
        s.task,
        e.session_id,
        e.timestamps,
        e.event,
        e.data
    FROM {READ_EVENTS} e
    JOIN sel s ON e.session_id = s._session_id
    WHERE CAST(e.subject_id AS VARCHAR) IN (SELECT subject_id FROM sel)
    ORDER BY s.subject_id, s.session_date, e.timestamps
""").df()

print(f"{len(df_events)} events from {df_events['session_id'].nunique()} sessions")
print(f"Event types: {sorted(df_events['event'].unique())}")
df_events.head(10)

## 4. Aggregate across sessions — all in SQL

In [ ]:
duckdb.sql(f"""
    WITH sel AS (
        SELECT _session_id, subject_id, session_date, task, foraging_eff
        FROM read_parquet('{SESSION_OUT}')
        WHERE task LIKE '%Uncoupled%'
          AND foraging_eff > 0.8
    )
    SELECT
        s.subject_id,
        COUNT(DISTINCT s._session_id)       AS n_sessions,
        COUNT(*)                            AS n_trials,
        AVG(t.earned_reward::DOUBLE)        AS mean_reward_rate,
        AVG(s.foraging_eff)                 AS mean_foraging_eff
    FROM {READ_TRIALS} t
    JOIN sel s ON t.session_id = s._session_id
    WHERE CAST(t.subject_id AS VARCHAR) IN (SELECT subject_id FROM sel)
    GROUP BY s.subject_id
    ORDER BY mean_foraging_eff DESC
""").df()